In [8]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress

from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'


#### Beware: deleted computed_shelfmelt_AIS_IMAU_UFEMISM1_expAE05.nc in ComputedScalarsHelene have orig_... and hb_calculated_now...

In [9]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = [] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame

In [10]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]
dic_area_of_interest = {}
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Amundsen'] = ['_sector_4']
y2sec=-365.25*24*3600
factor = y2sec/10**12

In [11]:
lst = glob.glob(pth_helene+f'expAE02/shelfmelt/*.nc')
lst.sort()


In [12]:
lst

['/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/ComputedScalarsHelene/expAE02/shelfmelt/computed_shelfmelt_AIS_DC_ISSM_expAE02.nc',
 '/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/ComputedScalarsHelene/expAE02/shelfmelt/computed_shelfmelt_AIS_DOE_MALI_4km_expAE02.nc',
 '/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/ComputedScalarsHelene/expAE02/shelfmelt/computed_shelfmelt_AIS_DOE_MALI_8km_Ant95_expAE02.nc',
 '/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/ComputedScalarsHelene/expAE02/shelfmelt/computed_shelfmelt_AIS_DOE_MALI_8km_AntMean_expAE02.nc',
 '/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/ComputedScalarsHelene/expAE02/shelfmelt/computed_shelfmelt_AIS_IGE_ElmerIce_expAE02.nc',
 '/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/ComputedScalarsHelene/expAE02/shelfmelt/computed_shelfmelt_AIS_ILTS_SICOPOLIS_expAE02.nc',
 '/Volumes/CrucialX8/computing/data/antarctica/ismip6_hackathon/Compute

In [13]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    #getting rid of old IMAU file (orig_...) and using the newly computed jb_computed... only:
    lst = lst[:-4]
    lst = lst[::-1]
    

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    
    #get shelf area
    pth_area = pth_way.replace('shelfmelt','iareafl')
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2100)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2100.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/2172666819.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/2172666819.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/2172666819.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [21]:
# avg and abs 2200
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2200)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2200.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/1456745597.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/1456745597.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/1456745597.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [22]:
# avg and abs 2300
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2300)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2300.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/2613283821.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/2613283821.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_17645/2613283821.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
